# Phase E: Customer Retention & Cohort Analysis
## Deep Dive: Repeat Purchase Dynamics & Customer Lifecycle
### Key Questions Explored:
1. What percentage of customers return to place a 2nd order?
2. How much revenue is driven by repeat customers vs one-time shoppers?
3. What is the month-over-month customer retention decay curve across cohorts?


In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')

DATA_DIR = '../data/processed'
orders = pd.read_csv(os.path.join(DATA_DIR, 'orders_enriched.csv'))
cohort_retention = pd.read_csv(os.path.join(DATA_DIR, 'cohort_retention_rate_matrix.csv'), index_col=0)
rfm = pd.read_csv(os.path.join(DATA_DIR, 'customer_rfm_segments.csv'))


### 1. First-Time vs. Repeat Customer Dynamics

In [ ]:
total_cust = len(rfm)
repeat_cust = (rfm['total_orders'] > 1).sum()
repeat_rate = (repeat_cust / total_cust) * 100

one_time_rev = rfm[rfm['total_orders'] == 1]['total_spent'].sum()
repeat_rev = rfm[rfm['total_orders'] > 1]['total_spent'].sum()
total_rev = one_time_rev + repeat_rev
repeat_rev_share = (repeat_rev / total_rev) * 100

print(f"Total Customer Population:    {total_cust:,}")
print(f"Repeat Customers (>= 2 orders): {repeat_cust:,} ({repeat_rate:.2f}%)")
print(f"One-Time Customer Revenue:    R$ {one_time_rev:,.2f} ({100-repeat_rev_share:.1f}%)")
print(f"Repeat Customer Revenue:      R$ {repeat_rev:,.2f} ({repeat_rev_share:.1f}%)")
print(f"Repeat Customer Spend Multiplier: {(repeat_rev/repeat_cust)/(one_time_rev/(total_cust-repeat_cust)):.2f}x higher spend per customer")


### 2. Monthly Cohort Retention Heatmap

In [ ]:
# Display heatmap for main active cohorts (2017 cohorts up to month 12)
subset_matrix = cohort_retention.loc['2017-01':'2017-12', [str(i) for i in range(13) if str(i) in cohort_retention.columns]]

plt.figure(figsize=(14, 8))
sns.heatmap(
    subset_matrix,
    annot=True,
    fmt=".1f",
    cmap="YlGnBu",
    vmin=0,
    vmax=5,
    cbar_kws={'label': 'Retention Rate (%)'}
)
plt.title('Monthly Customer Cohort Retention Rate (%) - 2017 Cohorts', fontsize=14, fontweight='bold', pad=15)
plt.xlabel('Cohort Index (Months Since First Purchase)', fontweight='bold')
plt.ylabel('Cohort Month', fontweight='bold')
plt.tight_layout()
plt.show()
